# <span style="color:#FF6B6B">Population</span> <span style="color:#4ECDC4">Chemotaxis</span> <span style="color:#45B7D1">Analysis:</span> <span style="color:#96CEB4">Group</span> <span style="color:#6C88C4">Behavior</span> <span style="color:#4F6196">Study</span>

## <span style="color:#1A365D">Overview</span>

This notebook processes and analyzes C. elegans population movement data, focusing on collective chemotactic behavior. The analysis builds upon individual worm tracking data generated from the population centerline pipeline.

### Getting Started
Step 1: Data Import and Preparation

In [ ]:
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm  # For Jupyter Notebook-compatible progress bar
import random
import pickle
import os

def process_chemotaxis_files(source_path: str, target_filename: str = "chemotaxis_params.csv", nan_threshold_percent: int = 40) -> dict:
    """
    Process chemotaxis CSV files from a specific folder structure and combine them.
    
    This function iterates over a multi-level folder structure to find and process CSV files.
    It reads each file with a two-row header, adds a 'trackID' column, and filters out DataFrames
    where the percentage of rows with any NaN values in the 'Spline_K' columns exceeds a specified threshold.
    
    Args:
        source_path (str): Path to the root source folder containing subfolders.
        target_filename (str): Name of CSV files to search for in each output folder (default: "chemotaxis_params.csv").
        nan_threshold_percent (float): Threshold percentage for allowable rows with NaN values in 'Spline_K' columns 
                                       before excluding the DataFrame (default: 50).
    
    Returns:
        dict: Dictionary with subsubfolder names as keys and concatenated DataFrames as values.
    """
    source_path = Path(source_path)
    results_dict = {}
    
    # Collect all first-level subdirectories
    subfolders = [subfolder for subfolder in source_path.iterdir() if subfolder.is_dir()]
    
    # Set up the progress bar
    with tqdm(total=len(subfolders), desc="Processing Subfolders") as pbar:
        for subfolder in subfolders:
            
            for subsubfolder in subfolder.iterdir():
                if not subsubfolder.is_dir():
                    continue
                
                subsubfolder_dfs = []  # List to store DataFrames for each subsubfolder
                
                for subsubsubfolder in subsubfolder.iterdir():
                    if not subsubsubfolder.is_dir():
                        continue
                    
                    output_folder = subsubsubfolder / "output"
                    
                    # Check if the output folder exists and contains the target file
                    target_file = output_folder / target_filename
                    if target_file.exists():
                        try:
                            # Read CSV with a two-row header and the first column as index
                            df = pd.read_csv(target_file, header=[0, 1], index_col=0)
                            
                            # Add the 'trackID' column with a MultiIndex
                            df[('trackID', 'trackID')] = subsubsubfolder.name
                            
                            # Extract 'Spline_K' columns
                            spline_k_df = df.loc[:, df.columns.get_level_values(0) == 'Spline_K']
                            
                            # Check if the percentage of rows with any NaN in 'Spline_K' columns is below the threshold
                            if spline_k_df.isna().any(axis=1).mean() * 100 < nan_threshold_percent:
                                subsubfolder_dfs.append(df)
                                
                        except Exception as e:
                            print(f"Error reading file {target_file}: {str(e)}")
                
                # Combine DataFrames if any were found for the current subsubfolder
                if subsubfolder_dfs:
                    combined_df = pd.concat(subsubfolder_dfs, axis=0)
                    
                    # Optional: Reorder columns to have 'trackID' first
                    # Uncomment the following lines if you want 'trackID' to be the first column
                    # cols = combined_df.columns.tolist()
                    # trackid_col = cols.pop(cols.index(('trackID', 'trackID')))
                    # combined_df = combined_df[[trackid_col] + cols]
                    
                    # Ensure each subsubfolder has its own entry in the dictionary
                    results_dict[subsubfolder.name] = combined_df
            
            # Update the progress bar
            pbar.update(1)
    
    return results_dict

# Example path - replace with your actual path
source_folder = "/lisc/scratch/neurobiology/zimmer/schaar/Behavior/High_Res_Population/elpiniki_data"

# Process the files with progress tracking
results_dict = process_chemotaxis_files(source_folder)

# Visualize results
for key, df in results_dict.items():
    print(f"--- {key} ---")
    print(f"Number of rows: {len(df)}")
    print(f"Number of columns: {df.shape[1]}")
    #print("Columns:", df.columns.tolist())
    #print(df.head())
    print("\n")


Processing Subfolders:   0%|          | 0/40 [00:00<?, ?it/s]

In [ ]:
# Visualize results per experiment (parsing experiment name from trackID)
for key, df in results_dict.items():
    # Extract experiment names by removing the track suffix
    df_copy = df.copy()
    df_copy['experiment'] = df_copy[('trackID', 'trackID')].str.rsplit('_track', n=1).str[0]
    
    # Group by experiment name
    experiment_groups = df_copy.groupby('experiment')
    
    print(f"\n=== {key} GROUP ===")
    
    for experiment_name, experiment_df in experiment_groups:
        unique_tracks = experiment_df[('trackID', 'trackID')].nunique()
        total_rows = len(experiment_df)
        columns = [col for col in df.columns.get_level_values(0).unique()]
        
        print(f"--- {experiment_name} ---")
        print(f"Number of tracks: {unique_tracks}")
        print(f"Total rows: {total_rows}")
        print(f"Columns: {columns}")
        print()

In [ ]:
# Visualize df columns
for key, df in results_dict.items():
    print("Columns:", df.columns.tolist())
    break

## <span style="color:#1A365D">Visualize Dataset Consistency Per Recording</span>

Use the dropdown to select recording folder of dataset and generate summary Kymograph as a proxy for dataquality

In [ ]:
# Enable Matplotlib Inline Backend
%matplotlib inline

# Import Necessary Libraries
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import time
import pandas as pd
import matplotlib.patches as mpatches
from ipywidgets import IntProgress, HTML, VBox  # Import progress bar widgets

# Define the Modified Plotting Function with Progress Bar and Parameter Printing
def plot_skeleton_spline_inline(spline_data, track_ids):
    '''
    Plots the spline_data dataframe as a kymogram inline in Jupyter Notebook,
    with an additional trackID indicator bar below each subplot and a progress bar.
    Additionally, prints parameters such as the number of unique tracks before plotting.
    
    :param spline_data: pandas DataFrame containing only 'Spline_K' columns
    :param track_ids: pandas Series containing 'trackID' corresponding to each frame
    :return: None
    '''
    try:
        start_time = time.time()
        num_frames = len(spline_data)

        if num_frames == 0:
            print("The selected DataFrame is empty. Please select a valid dataset.")
            return

        # Dynamically adjust number of rows based on number of frames
        if num_frames < 100000:
            num_lines = 10
        elif num_frames < 200000:
            num_lines = 20
        elif num_frames < 300000:
            num_lines = 30
        else:
            num_lines = 40

        # Optionally, limit the maximum number of subplots to prevent overcrowding
        max_subplots = 30
        num_lines = min(num_lines, max_subplots)

        cut_frames = num_frames // num_lines
        fig_height_per_plot = 2  # Adjusted height per plot

        # Initialize figure with constrained_layout
        fig, axs = plt.subplots(
            num_lines, 1,
            dpi=100,
            figsize=(14, fig_height_per_plot * num_lines * 1),
            constrained_layout=True  # Use constrained_layout instead of tight_layout
        )

        # Ensure axs is iterable
        if num_lines == 1:
            axs = [axs]

        # Create a color palette for trackIDs
        unique_track_ids = track_ids.unique()
        num_unique_tracks = len(unique_track_ids)
        cmap = plt.get_cmap('tab20')  # You can choose a different colormap if needed
        color_dict = {track_id: cmap(i % 20) for i, track_id in enumerate(unique_track_ids)}

        # Print number of unique tracks
        print(f"Number of unique tracks: {num_unique_tracks}")
        print(f"Be patient, might take a few minutes...")

        # Initialize Progress Bar and Status Label
        progress = IntProgress(
            min=0,
            max=num_lines,
            description='Progress:',
            bar_style='info'  # 'info', 'success', 'warning', 'danger'
        )
        status = HTML(value=f"Number of unique tracks: {num_unique_tracks}. Starting plotting...")
        progress_box = VBox([progress, status])  # Combine into a single widget
        display(progress_box)  # Display the progress bar

        for i, ax in enumerate(axs):
            start_idx = i * cut_frames
            end_idx = start_idx + cut_frames
            data_slice = spline_data.iloc[start_idx:end_idx].T
            track_slice = track_ids.iloc[start_idx:end_idx]

            if data_slice.empty:
                print(f"Data slice for subplot {i+1} is empty. Skipping...")
                progress.value = i + 1  # Update progress
                status.value = f"Skipped subplot {i+1}/{num_lines}"
                continue

            # Plot the kymogram
            cax = ax.imshow(
                data_slice,
                origin="upper",
                cmap='seismic',
                aspect='auto',
                vmin=-0.06,
                vmax=0.06
            )

            # Set ticks
            ax.set_xticks(np.linspace(0, cut_frames, 5))
            ax.set_xticklabels(np.linspace(start_idx, end_idx, 5).astype(int), fontsize=6, rotation=45)
            ax.set_yticks(ax.get_yticks())
            ax.set_yticklabels(ax.get_yticks(), fontsize=6)

            # Labeling
            if i == num_lines - 1:
                ax.set_xlabel('Frame', fontsize=6)
            ax.set_ylabel('Body Part', fontsize=6)

            # Add TrackID indicator bar
            ax_track = ax.inset_axes([0, -0.15, 1, 0.1], transform=ax.transAxes)
            ax_track.set_xlim(0, cut_frames)
            ax_track.set_ylim(0, 1)
            ax_track.axis('off')  # Hide the axis

            # Assign colors based on trackID
            for track_id in track_slice.unique():
                color = color_dict[track_id]
                mask = track_slice == track_id

                # Identify consecutive regions for the current track_id
                regions = []
                in_region = False
                start = 0
                for j, val in enumerate(mask):
                    if val and not in_region:
                        in_region = True
                        start = j
                    elif not val and in_region:
                        in_region = False
                        regions.append((start, j))
                if in_region:
                    regions.append((start, len(mask)))

                # Draw rectangles for each region
                for region in regions:
                    rect = mpatches.Rectangle(
                        (region[0], 0),
                        region[1]-region[0],
                        1,
                        color=color,
                        linewidth=0
                    )
                    ax_track.add_patch(rect)

            # Optionally, add a legend for trackIDs only once
            if i == 0:
                handles = [
                    mpatches.Patch(color=color_dict[tid], label=str(tid)) 
                    for tid in unique_track_ids
                ]
                # Place the legend outside the plot area to save space
                fig.legend(
                    handles=handles,
                    bbox_to_anchor=(1.05, 1),
                    loc='upper left',
                    fontsize=6
                )

            # Update Progress Bar and Status
            progress.value = i + 1
            status.value = f"Plotted subplot {i+1}/{num_lines}"

        # Show colorbar
        cbar = fig.colorbar(
            cax,
            ax=axs,
            orientation='vertical',
            fraction=0.02,
            pad=0.04
        )
        cbar.set_label('Intensity', fontsize=6)

        plt.show()

        # Hide the progress bar after completion
        progress_box.close()

        end_time = time.time()
        elapsed_time = end_time - start_time
        print(f"Plotting completed in {elapsed_time:.2f} seconds.")

    except Exception as e:
        print(f'Problem plotting the data: {e}')

# Define the Callback Function for the Dropdown
def on_dropdown_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        selected_key = change['new']
        selected_df = results_dict.get(selected_key, None)
        
        with output:
            clear_output(wait=True)
            if selected_df is not None:
                # Check if columns are MultiIndex and contain 'Spline_K'
                if isinstance(selected_df.columns, pd.MultiIndex):
                    # Get the first level of the MultiIndex
                    top_level = selected_df.columns.get_level_values(0)
                    if 'Spline_K' in top_level:
                        # Extract only the 'Spline_K' columns
                        spline_df = selected_df['Spline_K']
                        # Extract 'trackID' column as a Series
                        if ('trackID', 'trackID') in selected_df.columns:
                            track_id_series = selected_df[('trackID', 'trackID')]
                        else:
                            print(f"'trackID' column structure unexpected: {selected_df.columns}")
                            track_id_series = None
                        
                        if track_id_series is not None:
                            print(f"Selected Key: {selected_key}")
                            print(f"Extracted 'Spline_K' columns with shape {spline_df.shape}")
                            print("Loading plot...")
                            plot_skeleton_spline_inline(spline_df, track_id_series)
                        else:
                            print("Failed to extract 'trackID' as a Series.")
                    else:
                        print(f"'Spline_K' columns not found in the selected DataFrame.")
                else:
                    print("Selected DataFrame does not have MultiIndex columns.")
            else:
                print(f"No data found for key: {selected_key}")

# Create the Dropdown Widget
dropdown = widgets.Dropdown(
    options=list(results_dict.keys()),
    description='Select Data:',
    disabled=False,
)

# Create an Output Widget
output = widgets.Output()

# Link the Callback Function to the Dropdown
dropdown.observe(on_dropdown_change)

# Display the Dropdown and Output Widgets
display(dropdown, output)


<h2 style="color:#1A365D">Single Dataset Visualization</h2>

<h3>Single Dataset Overview</h3>
<p>Select a Dataset and get an overview about the assays tracks for centroid and nose.</p>



In [ ]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from ipywidgets import IntProgress, HTML, VBox

# Define the plotting function with adjustable density parameter and optional reversal marker
def plot_coordinates(df, density=10, show_reversal=False):
    """
    Plot 2D scatter plot of Centroid and Position 0 coordinates with adjustable density,
    optional reversal markers, and odor position.

    Parameters:
        df (pd.DataFrame): DataFrame containing 'X_rel_skel_pos_centroid', 'Y_rel_skel_pos_centroid',
                           'X_rel_skel_pos_0', 'Y_rel_skel_pos_0', 'odor_x', and 'odor_y' columns.
        density (int): Sampling interval for plotting every nth row.
        show_reversal (bool): Flag to indicate whether reversal markers should be displayed.
    """
    # Initialize progress bar
    progress = IntProgress(min=0, max=4, description='Plotting:', bar_style='info')
    status = HTML(value="Loading plot...")
    display(VBox([progress, status]))

    plt.figure(figsize=(10, 10))

    # Scatter plot for centroid coordinates with specified density and dark blue color
    plt.scatter(
        df[('chemotaxis_parameter', 'X_rel_skel_pos_centroid')][::density], 
        df[('chemotaxis_parameter', 'Y_rel_skel_pos_centroid')][::density], 
        label='Centroid Position', color='darkblue', s=0.05
    )
    progress.value += 1
    status.value = "Plotting Position 0 coordinates..."

    # Scatter plot for Position 0 coordinates with specified density and light blue color
    plt.scatter(
        df[('chemotaxis_parameter', 'X_rel_skel_pos_0')][::density], 
        df[('chemotaxis_parameter', 'Y_rel_skel_pos_0')][::density], 
        label='Position 0', color='lightblue', s=0.05
    )
    progress.value += 1
    status.value = "Plotting reversal onsets..." if show_reversal else "Plotting odor position..."

    # Optionally mark points where reversal onset is 1
    if show_reversal and ('chemotaxis_parameter', 'reversal_onset') in df.columns:
        reversal_points = df[df[('chemotaxis_parameter', 'reversal_onset')] == 1]
        plt.scatter(
            reversal_points[('chemotaxis_parameter', 'X_rel_skel_pos_centroid')],
            reversal_points[('chemotaxis_parameter', 'Y_rel_skel_pos_centroid')],
            color='red', marker='x', s=10, label='Reversal Onset'
        )
    progress.value += 1

    # Extract odor_x and odor_y from df
    odor_x = df[('chemotaxis_parameter', 'odor_x')].iloc[0]
    odor_y = df[('chemotaxis_parameter', 'odor_y')].iloc[0]

    # Plot odor position
    plt.scatter(odor_x, odor_y, color='green', marker='*', s=100, label='Odor Position')
    status.value = "Finalizing plot..."
    progress.value += 1

    # Setting labels, title, legend, and grid
    plt.xlabel('X Coordinate')
    plt.ylabel('Y Coordinate')
    plt.title(f'2D Scatter Plot of Centroid and Position 0 Coordinates (Every {density}th Row)')
    plt.legend()
    plt.grid(True)

    # Set x and y axis limits from 0 to 40.05
    plt.xlim(0, 40.05)
    plt.ylim(0, 40.05)

    plt.show()

    progress.close()  # Close progress bar when complete
    status.value = "Plot complete!"

# Callback for dropdown selection with adjustable density and reversal toggle
def on_dropdown_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        selected_key = change['new']
        selected_df = results_dict.get(selected_key, None)
        
        with plot_output:  # Use the unique output widget for plotting
            clear_output(wait=True)  # Clear previous output to avoid stacking
            if selected_df is not None:
                # Check if the necessary columns are present
                required_columns = [
                    ('chemotaxis_parameter', 'X_rel_skel_pos_centroid'),
                    ('chemotaxis_parameter', 'Y_rel_skel_pos_centroid'),
                    ('chemotaxis_parameter', 'X_rel_skel_pos_0'),
                    ('chemotaxis_parameter', 'Y_rel_skel_pos_0'),
                    ('chemotaxis_parameter', 'odor_x'),
                    ('chemotaxis_parameter', 'odor_y')
                ]
                if all(col in selected_df.columns for col in required_columns):
                    print(f"Selected Key: {selected_key}")
                    # Get density input and reversal display options
                    density = int(density_input.value)
                    show_reversal = reversal_checkbox.value
                    plot_coordinates(selected_df, density=density, show_reversal=show_reversal)
                else:
                    print("Required columns not found in selected DataFrame.")
            else:
                print(f"No data found for key: {selected_key}")

# Create dropdown, density input, and reversal checkbox widgets
dropdown = widgets.Dropdown(
    options=list(results_dict.keys()),
    description='Select Data:',
    disabled=False,
)
    
density_input = widgets.BoundedIntText(
    value=10,
    min=1,
    max=100,
    step=1,
    description='Density:',
    disabled=False
)

reversal_checkbox = widgets.Checkbox(
    value=False,
    description='Show Reversal Onset',
    disabled=False
)

# Create a unique output widget for the plotting block
plot_output = widgets.Output()

# Link the callback function to the dropdown
dropdown.observe(on_dropdown_change, names='value')

# Display the widgets and output
display(widgets.HBox([dropdown, density_input, reversal_checkbox]), plot_output)


# <span style="color:#FF6B6B; font-size:32px;">Random</span> <span style="color:#4ECDC4; font-size:32px;">Sampling</span> <span style="color:#45B7D1; font-size:32px;">of</span> <span style="color:#96CEB4; font-size:32px;">Whole</span> <span style="color:#6C88C4; font-size:32px;">Dataset</span>

## <span style="color:#1A365D; font-size:24px;">Overview</span>

Functions will randomly sample a percentage of the entire dataset into a new DataFrame, which can subsequently be used for plotting.


In [ ]:
def sample_and_summarize_all_columns(data_dict, sample_percent):
    """
    Sample random rows from each DataFrame in a dictionary and create a summary DataFrame with all columns.

    Parameters:
        data_dict (dict): A dictionary where keys are names and values are DataFrames.
        sample_percent (float): The percentage of rows to sample from each DataFrame.

    Returns:
        pd.DataFrame: Summary DataFrame with all columns and samples from each DataFrame in the dictionary.
    """
    # Dictionary to store sampled data
    sampled_data = {}

    for key, df in data_dict.items():
        # Calculate sample size
        sample_size = int(len(df) * (sample_percent / 100))
        
        # Sample the DataFrame
        sampled_df = df.sample(n=sample_size, random_state=random.randint(1, 100))
        
        # Store the sampled DataFrame in dictionary without adding the "Source" index
        sampled_data[key] = sampled_df

    # Concatenate all sampled DataFrames without creating a new "Source" index level
    summary_df = pd.concat(sampled_data).reset_index(drop=True)
    
    return summary_df




# Define your sample percentage
sample_percent = 100  # Sample 10% of rows from each DataFrame

# Get summary DataFrame with all columns
random_sampled_df = sample_and_summarize_all_columns(results_dict, sample_percent)

# One-liners to split the DataFrame
chemotaxis_sampled = random_sampled_df.loc[:, random_sampled_df.columns.get_level_values(0) == 'chemotaxis_parameter'].droplevel(0, axis=1)
spline_k_sampled = random_sampled_df.loc[:, random_sampled_df.columns.get_level_values(0) == 'Spline_K'].droplevel(0, axis=1)


In [ ]:
def analyze_chemotaxis_metrics(results_dict, save_plot=False, save_csv=False, output_path='./'):
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    from datetime import datetime
    
    metrics_dict = {
        'turns': [],
        'nan_count': [],
        'total_rows': [],
        'minutes': [],
        'df_names': []
    }
    
    for key, df in results_dict.items():
        turns_count = df[('chemotaxis_parameter', 'turn')][df[('chemotaxis_parameter', 'turn')] == 1].shape[0]
        total_rows = len(df)
        minutes = total_rows / (10 * 60)
        
        spline_cols = [col for col in df.columns if col[0] == 'Spline_K']
        nan_count = df[spline_cols].isna().all(axis=1).sum() if spline_cols else 0
        
        metrics_dict['turns'].append(turns_count)
        metrics_dict['nan_count'].append(nan_count)
        metrics_dict['total_rows'].append(total_rows)
        metrics_dict['minutes'].append(minutes)
        metrics_dict['df_names'].append(key)
    
    summary_df = pd.DataFrame({
        'DataFrame': metrics_dict['df_names'],
        'Total_Rows': metrics_dict['total_rows'],
        'Minutes': metrics_dict['minutes'],
        'Turns': metrics_dict['turns'],
        'NaN_Count': metrics_dict['nan_count']
    })
    
    summary_df['Turns_Percentage'] = (summary_df['Turns'] / summary_df['Total_Rows'] * 100).round(2)
    summary_df['NaN_Percentage'] = (summary_df['NaN_Count'] / summary_df['Total_Rows'] * 100).round(2)
    
    fig = plt.figure(figsize=(20, 6))
    gs = plt.GridSpec(1, 3)
    
    ax1 = fig.add_subplot(gs[0, 0])
    percentage_data = []
    for key, df in results_dict.items():
        total = len(df)
        turns_pct = df[('chemotaxis_parameter', 'turn')][df[('chemotaxis_parameter', 'turn')] == 1].shape[0] / total * 100
        nan_pct = (df[[col for col in df.columns if col[0] == 'Spline_K']].isna().all(axis=1).sum() / total * 100)
        
        percentage_data.append({
            'Dataset': key,
            'Turns': turns_pct,
            'NaN': nan_pct
        })
    
    event_df = pd.DataFrame(percentage_data)
    
    melted_df = pd.melt(event_df, id_vars=['Dataset'], 
                        value_vars=['Turns', 'NaN'],
                        var_name='Event Type', value_name='Percentage')
    sns.violinplot(data=melted_df, x='Event Type', y='Percentage', ax=ax1)
    
    for idx in range(len(event_df)):
        x_coords = [0, 1]
        y_coords = [event_df.iloc[idx]['Turns'], 
                   event_df.iloc[idx]['NaN']]
        ax1.plot(x_coords, y_coords, 'gray', alpha=0.3, linewidth=0.5)
    
    sns.swarmplot(data=melted_df, x='Event Type', y='Percentage', 
                  ax=ax1, color='black', alpha=0.5, size=4)
    
    ax1.set_title('Event Distribution')
    ax1.set_ylabel('Percentage of Frames')        
    
    ax2 = fig.add_subplot(gs[0, 1])
    time_data = pd.DataFrame({
        'Minutes': metrics_dict['minutes']
    })
    sns.violinplot(data=time_data, y='Minutes', inner='points', ax=ax2)
    ax2.set_title('Recording Duration Distribution')
    ax2.set_ylabel('Minutes')
    
    ax3 = fig.add_subplot(gs[0, 2])
    dataset_minutes = {key: len(df)/(10*60) for key, df in results_dict.items()}
    total_minutes = sum(dataset_minutes.values())
    
    sizes = [minutes for minutes in dataset_minutes.values()]
    
    n_colors = len(sizes)
    colors = [plt.cm.hsv(i/n_colors) for i in range(n_colors)]
    np.random.shuffle(colors)
    
    wedges, texts, autotexts = ax3.pie(sizes, colors=colors, 
                                      labels=[''] * len(sizes),
                                      autopct='')
    ax3.set_title(f'Dataset Size Distribution\nTotal Recording: {total_minutes:.1f} minutes')
    
    plt.tight_layout()
    
    if save_plot or save_csv:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        if save_plot:
            plot_path = f"{output_path}/chemotaxis_metrics_plot_{timestamp}.png"
            plt.savefig(plot_path, bbox_inches='tight', dpi=300)
            print(f"\nPlot saved to: {plot_path}")
        
        if save_csv:
            csv_path = f"{output_path}/chemotaxis_metrics_{timestamp}.csv"
            summary_df.to_csv(csv_path, index=False)
            print(f"CSV saved to: {csv_path}")
    
    plt.show()
    print("\nSummary Table (with percentages):")
    print(summary_df)
    
    return summary_df, event_df, fig

# Example usage:
summary_df, event_df, fig = analyze_chemotaxis_metrics(results_dict)


# Create output directory if it doesn't exist
os.makedirs('output', exist_ok=True)

# Save the outputs
fig.savefig('output/chemotaxis_metrics_plot.png', bbox_inches='tight', dpi=300)
summary_df.to_csv('output/chemotaxis_metrics.csv', index=False) 
event_df.to_csv('output/event_percentages.csv', index=False)

## <span style="color:#1A365D; font-size:24px;">save results_dict as pkl</span>

Save the Summary dictionary of all data as a pickle file

In [ ]:

# Define the path where you want to save the file
file_path = os.path.join(source_folder, 'results_dict.pkl')

# Save the results_dict
with open(file_path, 'wb') as f:
    pickle.dump(results_dict, f)


In [ ]:
# Visualize results
for key, df in results_dict.items():
    print(f"--- {key} ---")
    print(f"Number of rows: {len(df)}")
    print(f"Number of columns: {df.shape[1]}")
    print("Columns:", df.columns.tolist())
    #print(df.head())
    print("\n")
    break

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd

def create_interactive_plot_with_selector(results_dict):
    # Create dropdown for key selection
    key_dropdown = widgets.Dropdown(
        options=list(results_dict.keys()),
        description='Dataset:',
        style={'description_width': 'initial'}
    )
    
    # Calculate initial max slider value
    initial_df = results_dict[key_dropdown.value]
    total_frames = len(initial_df)
    max_slider = total_frames - 1
    
    # Create frame slider
    frame_slider = widgets.IntSlider(
        value=300,
        min=300,
        max=max_slider - 300,
        step=1,
        description='Current Frame:',
        continuous_update=False,
        style={'description_width': 'initial'}
    )
    
    # Output widget for plots
    out = widgets.Output()
    
    def update_slider_max(change):
        new_df = results_dict[change.new]
        new_max = len(new_df) - 1
        frame_slider.max = new_max - 300
        frame_slider.value = 300
        
    key_dropdown.observe(update_slider_max, names='value')
    
    def create_plots(current_frame):
        df = results_dict[key_dropdown.value]
        with out:
            clear_output(wait=True)
            
            # Calculate window range
            start_frame = max(0, current_frame - 300)
            end_frame = min(len(df), current_frame + 300)
            window = slice(start_frame, end_frame)
            
            # Create figure with subplots - adjust layout for better alignment
            fig = plt.figure(figsize=(15, 22), dpi=100)
            gs = fig.add_gridspec(9, 1, height_ratios=[2, 1, 1, 1, 1, 1, 1, 1, 1], hspace=0.4)
            
            # Create time array for x-axis
            time_points = np.arange(-300, 300) / 10
            
            # Plot 1: Spline Kymogram
            ax1 = fig.add_subplot(gs[0])
            spline_data = df.loc[:, ('Spline_K', slice(None))].iloc[window].T
            im = ax1.imshow(spline_data, aspect='auto', origin='upper', 
                           cmap='seismic', vmin=-0.06, vmax=0.06,
                           extent=[time_points[0], time_points[-1], spline_data.shape[0], 0])
            ax1.axvline(x=0, color='white', linestyle='--', alpha=0.5)
            ax1.set_xlabel('Time (seconds)')
            ax1.set_ylabel('Body Part')
            ax1.set_title('Spline Kymogram')
            
            # Plot 2: Binary Events
            ax2 = fig.add_subplot(gs[1])
            ax2.plot(time_points, df[('chemotaxis_parameter', 'turn')].iloc[window], 
                    label='Turn', drawstyle='steps-post', color='blue')
            ax2.plot(time_points, df[('chemotaxis_parameter', 'behaviour_state')].iloc[window], 
                    label='Behavior', drawstyle='steps-post', color='green')
            ax2.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
            ax2.set_ylabel('Event State')
            ax2.legend()
            ax2.set_title('Behavioral Events')
            ax2.grid(True, alpha=0.3)
            ax2.set_ylim(-1.2, 1.2)
            
            # Plot 3: Speed Centroid
            ax3 = fig.add_subplot(gs[2])
            ax3.plot(time_points, df[('chemotaxis_parameter', 'speed_centroid')].iloc[window], 
                    label='Speed Centroid', color='blue')
            ax3.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
            ax3.set_ylabel('Speed (mm/s)')
            ax3.set_title('Speed Centroid')
            ax3.grid(True, alpha=0.3)
            ax3.set_ylim(0, 0.2)  # Set speed limits from 0 to 0.2
            
            # Plot 4: Speed Center 24
            ax4 = fig.add_subplot(gs[3])
            ax4.plot(time_points, df[('chemotaxis_parameter', 'speed_center_24')].iloc[window], 
                    label='Speed Center 24', color='cyan')
            ax4.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
            ax4.set_ylabel('Speed (mm/s)')
            ax4.set_title('Speed Center 24')
            ax4.grid(True, alpha=0.3)
            ax4.set_ylim(0, 0.2)  # Set speed limits from 0 to 0.2
            
            # Plot 5: Navigation Index
            ax5 = fig.add_subplot(gs[4])
            ax5.plot(time_points, df[('chemotaxis_parameter', 'NI')].iloc[window], 
                    label='NI', color='red')
            ax5.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
            ax5.set_ylabel('Navigation Index')
            ax5.set_title('Navigation Index')
            ax5.grid(True, alpha=0.3)
            ax5.set_ylim(-1.2, 1.2)
            
            # Plot 6: Curving Angle
            ax6 = fig.add_subplot(gs[5])
            ax6.plot(time_points, df[('chemotaxis_parameter', 'curving_angle')].iloc[window], 
                    label='Curving Angle', color='purple')
            ax6.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
            ax6.set_ylabel('Angle (degrees)')
            ax6.set_title('Curving Angle')
            ax6.grid(True, alpha=0.3)
            
            # Plot 7: Bearing Angle
            ax7 = fig.add_subplot(gs[6])
            bearing_angle_data = df[('chemotaxis_parameter', 'bearing_angle')].iloc[window]
            abs_bearing = np.abs(bearing_angle_data)
            abs_bearing = np.minimum(abs_bearing, 182)
            ax7.plot(time_points, abs_bearing,
                    label='|Bearing Angle|', color='blue')
            ax7.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
            ax7.set_ylabel('Angle (degrees)')
            ax7.set_title('Bearing Angle')
            ax7.legend()
            ax7.grid(True, alpha=0.3)
            ax7.set_ylim(0, 182)
            
            # Plot 8: Distance to Odor
            ax8 = fig.add_subplot(gs[7])
            ax8.plot(time_points, df[('chemotaxis_parameter', 'distance_to_odor_centroid')].iloc[window], 
                    label='Distance to Odor', color='brown')
            ax8.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
            ax8.set_ylabel('Distance (mm)')
            ax8.set_title('Distance to Odor')
            ax8.grid(True, alpha=0.3)
            
            # Plot 9: Concentrations
            ax9 = fig.add_subplot(gs[8])
            ax9.plot(time_points, df[('chemotaxis_parameter', 'conc_at_0')].iloc[window], 
                    label='Conc at 0', color='orange')
            ax9.plot(time_points, df[('chemotaxis_parameter', 'conc_at_centroid')].iloc[window], 
                    label='Conc at Centroid', color='green')
            ax9.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
            ax9.set_ylabel('Concentration (mol)')
            ax9.set_xlabel('Time (seconds)')
            ax9.legend()
            ax9.set_title('Concentration Measurements')
            ax9.grid(True, alpha=0.3)
            
            # Adjust layout to ensure all plots are aligned
            plt.subplots_adjust(right=0.95)
            
            plt.show()
            
            # Display information
            print(f"Dataset: {key_dropdown.value}")
            print(f"Current frame: {current_frame} (Time = 0s)")
            print(f"Window: {start_frame} to {end_frame} (Time = -30s to +30s)")
            if ('trackID', 'trackID') in df.columns:
                track_ids = df[('trackID', 'trackID')].iloc[window].unique()
                print(f"Track IDs in window: {track_ids}")
    
    # Create interactive widget
    interactive_plot = widgets.interactive(create_plots, current_frame=frame_slider)
    
    # Layout all widgets
    controls = widgets.VBox([key_dropdown, frame_slider])
    full_widget = widgets.VBox([controls, out])
    
    return full_widget

# Example usage:
viewer = create_interactive_plot_with_selector(results_dict)
display(viewer)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import matplotlib.animation as animation
from pathlib import Path
import datetime

def create_video_export_viewer(results_dict):
    # Create dropdown for dataset selection
    key_dropdown = widgets.Dropdown(
        options=list(results_dict.keys()),
        description='Dataset:',
        style={'description_width': 'initial'}
    )
    
    # Create dropdown for track ID selection (will be updated based on dataset)
    track_dropdown = widgets.Dropdown(
        description='Track ID:',
        style={'description_width': 'initial'}
    )
    
    # Create export button
    export_button = widgets.Button(
        description='Export Video',
        button_style='success',
        tooltip='Click to create video animation'
    )
    
    # Create progress bar
    progress = widgets.FloatProgress(
        value=0,
        min=0,
        max=100,
        description='Progress:',
        bar_style='info',
        style={'bar_color': '#1976d2'},
        orientation='horizontal'
    )
    progress.layout.visibility = 'hidden'
    
    # Status output
    status_output = widgets.Output()
    
    def update_track_options(change):
        """Update track ID options when dataset changes"""
        df = results_dict[change.new]
        if ('trackID', 'trackID') in df.columns:
            track_ids = sorted(df[('trackID', 'trackID')].unique())
            track_dropdown.options = track_ids
            if track_ids:
                track_dropdown.value = track_ids[0]
        with status_output:
            clear_output()
            print(f"Selected dataset: {change.new}")
            print(f"Available tracks: {len(track_dropdown.options)}")
    
    key_dropdown.observe(update_track_options, names='value')
    
    def create_frame(frame_num, df, fig, axes, total_frames):
        progress.value = (frame_num - df.index[0]) / len(df) * 100
        
        # Calculate window range
        current_frame = frame_num
        start_frame = max(df.index[0], current_frame - 300)
        end_frame = min(df.index[-1], current_frame + 300)
        window = slice(start_frame, end_frame)
        
        time_points = np.arange(-300, 300) / 10
        
        # Clear all axes
        for ax in axes:
            ax.clear()
            
        # Plot 1: Turn Kymogram (previously Spline Kymogram)
        turn_data = df.loc[:, ('turn', slice(None))].iloc[window].T
        im = axes[0].imshow(turn_data, aspect='auto', origin='upper', 
                          cmap='seismic', vmin=-0.06, vmax=0.06,
                          extent=[time_points[0], time_points[-1], turn_data.shape[0], 0])
        axes[0].axvline(x=0, color='white', linestyle='--', alpha=0.5)
        axes[0].set_title('Turn Kymogram')
        
        # Plot 2: Binary Events (simplified)
        events_data = df[('chemotaxis_parameter', 'behaviour_state')].iloc[window]
        axes[1].plot(time_points[:len(events_data)], events_data, 'g-', label='Behavior')
        axes[1].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
        axes[1].set_title('Behavior State')
        
        # Plot 3: Speed
        speed_data = df[('chemotaxis_parameter', 'speed')].iloc[window]
        axes[2].plot(time_points[:len(speed_data)], speed_data, 'b-')
        axes[2].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
        axes[2].set_title('Speed')
        
        # Plot 4: Navigation Index
        ni_data = df[('chemotaxis_parameter', 'NI')].iloc[window]
        axes[3].plot(time_points[:len(ni_data)], ni_data, 'r-')
        axes[3].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
        axes[3].set_title('Navigation Index')
        
        # Plot 5: Curving Angle
        curve_data = df[('chemotaxis_parameter', 'curving_angle')].iloc[window]
        axes[4].plot(time_points[:len(curve_data)], curve_data, 'purple')
        axes[4].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
        axes[4].set_title('Curving Angle')
        
        # Plot 6: Bearing Angle
        bearing_data = df[('chemotaxis_parameter', 'bearing_angle')].iloc[window]
        axes[5].plot(time_points[:len(bearing_data)], bearing_data, 'g-')
        axes[5].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
        axes[5].set_title('Bearing Angle')
        
        # Plot 7: Distance to Odor
        dist_data = df[('chemotaxis_parameter', 'distance_to_odor_centroid')].iloc[window]
        axes[6].plot(time_points[:len(dist_data)], dist_data, 'brown')
        axes[6].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
        axes[6].set_title('Distance to Odor')
        
        # Plot 8: Concentrations (simplified)
        conc_data = df[('chemotaxis_parameter', 'conc_at_0')].iloc[window]
        axes[7].plot(time_points[:len(conc_data)], conc_data, 'orange')
        axes[7].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
        axes[7].set_title('Concentration')
        
        # Add frame number and percentage
        percentage = (frame_num - df.index[0]) / len(df) * 100
        fig.suptitle(f'Frame: {frame_num} ({percentage:.1f}%) - Track {track_dropdown.value}', y=0.995)
        
    def export_video(b):
        with status_output:
            clear_output()
            print(f"Starting video export for dataset: {key_dropdown.value}, track: {track_dropdown.value}")
            
            progress.layout.visibility = 'visible'
            progress.value = 0
            
            # Get full dataset and filter for selected track
            full_df = results_dict[key_dropdown.value]
            df = full_df[full_df[('trackID', 'trackID')] == track_dropdown.value].copy()
            
            if len(df) == 0:
                print("No data found for selected track!")
                progress.layout.visibility = 'hidden'
                return
                
            print(f"Track length: {len(df)} frames")
            
            # Create figure with reduced size and DPI
            fig = plt.figure(figsize=(10, 14), dpi=72)
            gs = fig.add_gridspec(8, 1, height_ratios=[2, 1, 1, 1, 1, 1, 1, 1], hspace=0.4)
            axes = [fig.add_subplot(g) for g in gs]
            
            # Create animation for all frames (no frame skipping)
            frames = df.index
            
            anim = animation.FuncAnimation(
                fig, create_frame,
                frames=frames,
                fargs=(df, fig, axes, len(df)),
                interval=100  # Set interval to adjust video speed (ms per frame)
            )
            
            Path("videos").mkdir(exist_ok=True)
            timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"videos/{key_dropdown.value}_track{track_dropdown.value}_{timestamp}.mp4"  # Save as MP4
            
            # Use 'ffmpeg' writer for video output
            writer = animation.FFMpegWriter(fps=30)  # Set FPS for smooth video playback
            
            anim.save(filename, writer=writer)
            
            print(f"Video exported to {filename}")
            plt.close()
            
            progress.layout.visibility = 'hidden'
            progress.value = 0
    
    export_button.on_click(export_video)
    
    # Initialize track dropdown with first dataset's tracks
    update_track_options(type('Change', (), {'new': key_dropdown.value})())
    
    # Layout all widgets
    controls = widgets.VBox([
        widgets.HBox([key_dropdown, track_dropdown]),
        export_button,
        progress,
        status_output
    ])
    
    return controls

# Example usage:
video_exporter = create_video_export_viewer(results_dict)
display(video_exporter)